In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.special import gammaln
import sys
from scipy.ndimage import gaussian_filter1d
from scipy.stats import wilcoxon, ranksums
from concurrent import futures 
# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

import importlib
import session_class
import cell_analysis
import multi_session_pca
importlib.reload(session_class)
importlib.reload(cell_analysis)
importlib.reload(multi_session_pca)

from session_class import Session
from cell_analysis import Cell
from multi_session_pca import MultiSessionPCA

import holoviews as hv
from holoviews import opts
from bokeh.io import output_notebook
import hvplot.pandas  # noqa: F401
output_notebook()
hv.extension('bokeh')

print("Imports loaded successfully!")

Loading BokehJS ...

Imports loaded successfully!


In [17]:
base_path = Path.cwd().parents[1] / 'data' / 'unified_cell_trial_data'
pickle_file = base_path / 'msn_fiona_cell_trial_data.pkl'

# Load MSN cell database
cell_df = pd.read_pickle(pickle_file)


In [38]:
print(cell_df[['cell_type', 'cell_ID']].drop_duplicates('cell_ID')['cell_type'].value_counts())

cell_df['grade'].value_counts()


cell_type
pu msn    861
msn       553
Name: count, dtype: int64


grade
8    584982
7    161227
6     24496
Name: count, dtype: int64

In [19]:
# Configuration
SSRT = 50  # Stop Signal Reaction Time in ms
SMOOTH_SIGMA = 15  # Smoothing sigma for gaussian filter (ms)
MIN_GRADE = 8   # Minimum cell quality grade

# Epoch parameters
BIN_SIZE = 1      # ms
EPOCH_WINDOW = [-200, 600] # ms around alignment event

"""
From Indra et al., 2020, methods section page 5;380:
The recording cylinder was implanted on the left hemisphere for monkey F(iona)
"""
CONTRA_DIR = 0  # Contra direction for monkey Fiona

print(f"Configuration set: SSRT={SSRT}ms, Grade>={MIN_GRADE}")


Configuration set: SSRT=50ms, Grade>=8


In [20]:
cell_id = 1547
cell_df['cell_ID'] ==cell_id
cell = Cell(cell_df[cell_df['cell_ID'] == cell_id])

print(cell.data['trial_number'].nunique() == cell.data.shape[0])
print(cell.data.shape)

cell.data.columns

True
(685, 27)


Index(['cell_ID', 'cell_type', 'maestro_ID', 'problem', 'grade', 'filename',
       'trial_name', 'reaction_time', 'go_cue', 'stop_cue', 'trial_failed',
       'ssd_len', 'ssd_number', 'type', 'first_relevant_saccade',
       'segs_durations', 'segs_times', 'trial_length', 'screen_rotation',
       'saccades', 'blinks', 'dir', 'neural_data', 'session', 'plexon_session',
       'trial_number', 'trial_session'],
      dtype='object')

In [21]:
cell.align_spikes_to_event(
    alignment_point='first_relevant_saccade', verbose=True
)

cell.align_spikes_to_event(
    alignment_point='go_cue', verbose=True
)

cell.data.columns

Index(['cell_ID', 'cell_type', 'maestro_ID', 'problem', 'grade', 'filename',
       'trial_name', 'reaction_time', 'go_cue', 'stop_cue', 'trial_failed',
       'ssd_len', 'ssd_number', 'type', 'first_relevant_saccade',
       'segs_durations', 'segs_times', 'trial_length', 'screen_rotation',
       'saccades', 'blinks', 'dir', 'neural_data', 'session', 'plexon_session',
       'trial_number', 'trial_session',
       'spikes_aligned_to_first_relevant_saccade', 'spikes_aligned_to_go_cue'],
      dtype='object')

In [22]:
cell.plot_raster(
    epok=[-500, 200],
    alignment_point='first_relevant_saccade',
    # alignment_point='go_cue',
    trial_type='GO',
    success_only=False,
    direction=CONTRA_DIR,
)

:Overlay
   .Curve.I   :Curve   [x]   (y)
   .HeatMap.I :HeatMap   [time,index]   (value)

In [26]:
def get_trial_spikes_for_Indra_condition_1(trial: pd.Series):
    """Count the amount of spikes"""
    assert trial['type'] == 'GO'
    assert trial['dir'] == CONTRA_DIR
    assert trial['trial_failed'] == False

    baseline_spikes_count = (
        (trial['spikes_aligned_to_go_cue'] >= -300) & 
        (trial['spikes_aligned_to_go_cue'] <= 0)
    ).sum()

    saccade_spikes_count = (
        (trial['spikes_aligned_to_first_relevant_saccade'] >= -100) & 
        (trial['spikes_aligned_to_first_relevant_saccade'] <= 100)
    ).sum()

    return baseline_spikes_count, saccade_spikes_count

def get_cell_spike_counts_for_Indra_condition_1(cell: Cell):
    """Count the amount of spikes for all legal trials of the cell"""
    cell.align_spikes_to_event(
        alignment_point='first_relevant_saccade', verbose=True
    )

    cell.align_spikes_to_event(
        alignment_point='go_cue', verbose=True
    )

    tmp_df = cell.data[
        (cell.data['type'] == 'GO') &
        (cell.data['dir'] == CONTRA_DIR) &
        (cell.data['reaction_time'] >= 180) &  # Slow-GO trials only
        (cell.data['trial_failed'] == False)
    ]

    spike_counts = tmp_df.apply(get_trial_spikes_for_Indra_condition_1, axis=1, result_type='expand').to_numpy()
    base_line_counts = spike_counts[:, 0]
    saccade_counts = spike_counts[:, 1]

    return base_line_counts, saccade_counts

def run_signed_rank_test_for_cell_Indra_condition_1(cell: Cell):
    """
    "GO" neurons were defined based on two criteria. The first: 
    (1) a significant firing rate increase around contralateral saccade onset 
        (-100 ms to +100 ms from saccade onset versus baseline from -300 ms to go) 
        on the go trials (one-tailed Wilcoxon signed-rank, P ≤ 0.05)
    """
    base_line_counts, saccade_counts = get_cell_spike_counts_for_Indra_condition_1(cell)

    stat, p_value = wilcoxon(
        base_line_counts,
        saccade_counts,
        alternative='less'  # Test if baseline < saccade (i.e., saccade firing rate increased)
    )
    return stat, p_value



trial = cell.data.iloc[212]
print(trial['spikes_aligned_to_go_cue'])
print(get_trial_spikes_for_Indra_condition_1(trial))

cell_dist = get_cell_spike_counts_for_Indra_condition_1(cell)
print(f"Cell spike counts shape: {cell_dist[0].shape}, {cell_dist[1].shape}")

stat, p_value = run_signed_rank_test_for_cell_Indra_condition_1(cell)
print(f"Wilcoxon signed-rank test result: stat={stat}, p-value={p_value}")

[-842.57 -114.08  -15.22]
(np.int64(2), np.int64(0))
Cell spike counts shape: (126,), (126,)
Wilcoxon signed-rank test result: stat=1413.0, p-value=0.999999998937877


In [27]:
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm

def process_cell(cell_id: int) -> tuple:
    """Process a single cell and return its signed rank test results"""
    try:
        cell_data = cell_df[cell_df['cell_ID'] == cell_id]
        cell_obj = Cell(cell_data)
        stat, p_value = run_signed_rank_test_for_cell_Indra_condition_1(cell_obj)
        return cell_id, stat, p_value, None
    except Exception as e:
        return cell_id, np.nan, np.nan, str(e)

# Get unique cell IDs
unique_cell_ids = cell_df['cell_ID'].unique()
print(f"Processing {len(unique_cell_ids)} cells...")

# Process cells in parallel
results = []
with futures.ProcessPoolExecutor() as executor:
    future_to_cell = {executor.submit(process_cell, cell_id): cell_id for cell_id in unique_cell_ids}
    for future in tqdm(futures.as_completed(future_to_cell), total=len(unique_cell_ids), desc="Processing Cells"):
        result = future.result()
        results.append(result)

# Create DataFrame from results
results_df = pd.DataFrame(results, columns=['cell_ID', 'stat', 'p_value', 'error'])
results_df = results_df.sort_values('cell_ID').reset_index(drop=True)

# Apply Holm-Bonferroni correction (only for non-NaN p-values)
valid_mask = ~results_df['p_value'].isna()
results_df['p_value_holm_bonferroni'] = np.nan

if valid_mask.sum() > 0:
    reject, pvals_corrected, _, _ = multipletests(
        results_df.loc[valid_mask, 'p_value'], 
        method='holm'
    )
    results_df.loc[valid_mask, 'p_value_holm_bonferroni'] = pvals_corrected
    results_df.loc[valid_mask, 'significant_holm'] = reject

results_df['regular_rejected'] = results_df['p_value'] < 0.05

print(f"\nCompleted! {valid_mask.sum()} cells processed successfully, {(~valid_mask).sum()} errors")
print(f"Significant cells (Holm-Bonferroni corrected, p<0.05): {results_df['significant_holm'].sum()}")

significant_cell_ids = set(results_df[results_df['regular_rejected'] == True]['cell_ID'])
print(f"Number of significant cells: {len(significant_cell_ids)}")

results_df.head(10)

Processing 1414 cells...


Processing Cells: 100%|██████████| 1414/1414 [00:01<00:00, 1040.71it/s]



Completed! 1413 cells processed successfully, 1 errors
Significant cells (Holm-Bonferroni corrected, p<0.05): 59
Number of significant cells: 137


,cell_ID,stat,p_value,error,p_value_holm_bonferroni,significant_holm,regular_rejected
0,11,98.0,0.999192,None,1.0,False,False
1,15,40.0,0.917241,None,1.0,False,False
2,16,237.5,0.997867,None,1.0,False,False
3,17,1149.0,1.000000,None,1.0,False,False
4,18,217.5,0.500000,None,1.0,False,False
5,19,108.0,0.862383,None,1.0,False,False
6,20,20.0,0.369441,None,1.0,False,False
7,21,549.0,0.997101,None,1.0,False,False
8,23,215.0,0.998954,None,1.0,False,False
9,24,72.5,0.912285,None,1.0,False,False


In [28]:
rnd_cell_id = np.random.choice(list(significant_cell_ids))
print(f"Selected Cell ID: {rnd_cell_id}")

cell_plot = Cell(cell_df[cell_df['cell_ID'] == rnd_cell_id])
SMOOTH_SIGMA = 15
saccade_plot = cell_plot.plot_psth_by_type_direction(
    alignment_point='first_relevant_saccade',
    epok=[-100, 100],
    bin_size=1,
    smooth_ker_size=SMOOTH_SIGMA,
)

baseline_plot = cell_plot.plot_psth_by_type_direction(
    alignment_point='go_cue',
    epok=[-400, 0],
    bin_size=1,
    smooth_ker_size=SMOOTH_SIGMA,
)

(
    saccade_plot[CONTRA_DIR]['GO'].opts(height=200, width=400) + 
    baseline_plot[CONTRA_DIR]['GO'].opts(opts.Curve(color='orange')).opts(height=200, width=400)
) #.cols(1)

Selected Cell ID: 9543


:Layout
   .Overlay.I  :Overlay
      .Curve.I :Curve   [Time]   (Firing Rate (spikes/s))
      .VLine.I :VLine   [x,y]
   .Overlay.II :Overlay
      .Curve.I :Curve   [Time]   (Firing Rate (spikes/s))
      .VLine.I :VLine   [x,y]

In [29]:
cell.data
# cell.align_spikes_to_event(alignment_point='stop_cue', verbose=False)
cell.data[cell.data['type'] == 'GO']

,cell_ID,cell_type,maestro_ID,problem,grade,filename,trial_name,reaction_time,go_cue,stop_cue,...,saccades,blinks,dir,neural_data,session,plexon_session,trial_number,trial_session,spikes_aligned_to_first_relevant_saccade,spikes_aligned_to_go_cue
152,1547,msn,7,NaN,8,fi211025a.0738,GO_R,259.0,1008,NaN,...,"[[355, 414], [1267, 1340]]",None,0,[804.15],fi211025,a,738,fi211025a,[-462.85],[-203.85000000000002]
153,1547,msn,7,NaN,8,fi211025a.0542,GO_R,163.0,1016,NaN,...,"[[0, 54], [431, 480], [457, 493], [1179, 1253]...",None,0,[1045.92],fi211025,a,542,fi211025a,[-133.07999999999993],[29.920000000000073]
154,1547,msn,7,NaN,8,fi211025a.0220,GO_R,127.0,1074,NaN,...,"[[353, 411], [1201, 1276], [1245, 1280], [1252...",None,0,[2067.37],fi211025,a,220,fi211025a,[866.3699999999999],[993.3699999999999]
155,1547,msn,7,NaN,8,fi211025a.0626,GO_R,195.0,999,NaN,...,"[[61, 137], [168, 287], [257, 317], [601, 661]...","[106, 188]",0,[2025.72],fi211025,a,626,fi211025a,[831.72],[1026.72]
156,1547,msn,7,NaN,8,fi211025a.0660,GO_R,40.0,1052,NaN,...,"[[84, 159], [1092, 1148], [1290, 1364], [2154,...",None,0,"[168.53, 442.32, 449.2, 851.8]",fi211025,a,660,fi211025a,"[-923.47, -649.6800000000001, -642.8, -240.200...","[-883.47, -609.6800000000001, -602.8, -200.200..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
534,1547,msn,7,NaN,8,fi211025a.0386,GO_L,182.0,1080,NaN,...,"[[105, 181], [1262, 1338], [2154, 2184]]",None,180,"[1178.0, 1309.6, 1320.07]",fi211025,a,386,fi211025a,"[-84.0, 47.59999999999991, 58.069999999999936]","[98.0, 229.5999999999999, 240.06999999999994]"
535,1547,msn,7,NaN,8,fi211025a.0617,GO_L,137.0,1093,NaN,...,"[[132, 213], [787, 848], [1230, 1302], [1270, ...",None,180,"[490.57, 1233.82, 1234.92, 1278.45, 1337.3, 13...",fi211025,a,617,fi211025a,"[-739.4300000000001, 3.8199999999999363, 4.920...","[-602.4300000000001, 140.81999999999994, 141.9..."
536,1547,msn,7,NaN,8,fi211025a.0691,GO_L,235.0,902,NaN,...,"[[101, 176], [686, 746], [1137, 1212], [1587, ...",None,180,"[27.45, 1148.87, 1155.1, 1167.57, 1184.37]",fi211025,a,691,fi211025a,"[-1109.55, 11.86999999999989, 18.0999999999999...","[-874.55, 246.8699999999999, 253.0999999999999..."
537,1547,msn,7,NaN,8,fi211025a.0711,GO_L,156.0,1041,NaN,...,"[[202, 280], [743, 804], [1197, 1271], [1239, ...",None,180,"[262.09, 1173.01, 1178.11, 1524.81, 1543.21, 1...",fi211025,a,711,fi211025a,"[-934.9100000000001, -23.99000000000001, -18.8...","[-778.9100000000001, 132.01, 137.1099999999999..."


In [30]:
"""
(2) a significant decrease in correct stop trials vs. slow go in the contralateral direction (in
0-400 ms bin from stop signal onset) (one-tailed Wilcoxon signed-rank, P ≤ 0.05)

Note: Using rank-sum (Mann-Whitney U) test since we're comparing two independent groups
(correct stop trials vs slow go trials), not paired samples.
"""
from scipy.stats import ranksums

def get_trial_spikes_for_Indra_condition_2(trial: pd.Series, trial_type: str):
    """Count spikes in 0-400ms window from stop signal onset"""
    assert trial['dir'] == CONTRA_DIR
    assert trial['trial_failed'] == False
    
    # Spikes aligned to stop cue (for GO trials, stop_cue is set to go_cue + alignment_bias)
    spikes = trial['spikes_aligned_to_stop_cue']
    
    spike_count = ((spikes >= 0) & (spikes <= 400)).sum()
    return spike_count

def get_cell_spike_counts_for_Indra_condition_2(cell: Cell):
    """Get spike counts for slow GO and correct STOP trials"""
    cell.align_spikes_to_event(alignment_point='stop_cue', verbose=False)
    
    # Slow GO trials (RT >= 150ms, successful, contralateral)
    slow_go_df = cell.data[
        (cell.data['type'] == 'GO') &
        (cell.data['dir'] == CONTRA_DIR) &
        (cell.data['trial_failed'] == False) &
        (cell.data['reaction_time'] >= 150)
    ]
    
    # Correct STOP trials (successful, contralateral)
    correct_stop_df = cell.data[
        (cell.data['type'] == 'STOP') &
        (cell.data['dir'] == CONTRA_DIR) &
        (cell.data['trial_failed'] == False)
    ]
    
    slow_go_counts = slow_go_df.apply(
        lambda x: get_trial_spikes_for_Indra_condition_2(x, 'GO'), axis=1
    ).to_numpy()
    
    correct_stop_counts = correct_stop_df.apply(
        lambda x: get_trial_spikes_for_Indra_condition_2(x, 'STOP'), axis=1
    ).to_numpy()
    
    return slow_go_counts, correct_stop_counts

def run_ranksum_test_for_cell_Indra_condition_2(cell: Cell):
    """
    Test for significant decrease in correct stop trials vs. slow go trials.
    Uses rank-sum (Mann-Whitney U) test since groups are independent.
    
    alternative='less' tests if correct_stop < slow_go (i.e., decrease in stop trials)
    """
    slow_go_counts, correct_stop_counts = get_cell_spike_counts_for_Indra_condition_2(cell)
    
    if len(slow_go_counts) < 5 or len(correct_stop_counts) < 5:
        return np.nan, np.nan  # Not enough trials
    
    stat, p_value = ranksums(
        correct_stop_counts,
        slow_go_counts,
        # alternative='less'  # Test if correct_stop < slow_go (decrease in stop trials)
    )
    return stat, p_value

# Test on the example cell
slow_go_counts, correct_stop_counts = get_cell_spike_counts_for_Indra_condition_2(cell)
print(f"Slow GO trials: {len(slow_go_counts)}, mean spikes: {slow_go_counts.mean():.2f}")
print(f"Correct STOP trials: {len(correct_stop_counts)}, mean spikes: {correct_stop_counts.mean():.2f}")

stat, p_value = run_ranksum_test_for_cell_Indra_condition_2(cell)
print(f"Rank-sum test result: stat={stat:.3f}, p-value={p_value:.4f}")

Slow GO trials: 153, mean spikes: 0.09
Correct STOP trials: 35, mean spikes: 0.40
Rank-sum test result: stat=2.366, p-value=0.0180


In [31]:
def process_cell_condition_2(cell_id: int) -> tuple:
    """Process a single cell and return its signed rank test results"""
    try:
        cell_data = cell_df[cell_df['cell_ID'] == cell_id]
        ssd_mean = int(np.nanmean(cell_data[cell_data['ssd_number'] <= 3]['ssd_len']))
        cell_obj = Cell(cell_data, go_trials_stop_cue_alignment_bias=ssd_mean)
        stat, p_value = run_ranksum_test_for_cell_Indra_condition_2(cell_obj)
        return cell_id, stat, p_value, None
    except Exception as e:
        return cell_id, np.nan, np.nan, str(e)

# Get unique cell IDs
unique_cell_ids = cell_df['cell_ID'].unique()
print(f"Processing {len(unique_cell_ids)} cells...")

# Process cells in parallel
results_cond_2 = []
with futures.ProcessPoolExecutor() as executor:
    future_to_cell_cond_2 = {executor.submit(process_cell_condition_2, cell_id): cell_id for cell_id in unique_cell_ids}
    for future in tqdm(futures.as_completed(future_to_cell_cond_2), total=len(unique_cell_ids), desc="Processing Cells for condition #2"):
        result = future.result()
        results_cond_2.append(result)

# Create DataFrame from results
results_cond_2_df = pd.DataFrame(results_cond_2, columns=['cell_ID', 'stat', 'p_value', 'error'])
results_cond_2_df = results_cond_2_df.sort_values('cell_ID').reset_index(drop=True)

# Apply Holm-Bonferroni correction (only for non-NaN p-values)
valid_mask = ~results_cond_2_df['p_value'].isna()
results_cond_2_df['p_value_holm_bonferroni'] = np.nan

if valid_mask.sum() > 0:
    reject, pvals_corrected, _, _ = multipletests(
        results_cond_2_df.loc[valid_mask, 'p_value'], 
        method='holm'
    )
    results_cond_2_df.loc[valid_mask, 'p_value_holm_bonferroni'] = pvals_corrected
    results_cond_2_df.loc[valid_mask, 'significant_holm'] = reject
results_cond_2_df['regular_rejected'] = results_cond_2_df['p_value'] < 0.05

print(f"\nCompleted! {valid_mask.sum()} cells processed successfully, {(~valid_mask).sum()} errors")
print(f"Significant cells (Holm-Bonferroni corrected, p<0.05): {results_cond_2_df['significant_holm'].sum()}")
cond_2_significant_cell_ids = set(results_cond_2_df[results_cond_2_df['significant_holm'] == True]['cell_ID'])
print(f"Number of significant cells for condition #2: {len(cond_2_significant_cell_ids)}")

results_cond_2_df.head(10)

Processing 1414 cells...


Processing Cells for condition #2: 100%|██████████| 1414/1414 [00:00<00:00, 1828.12it/s]



Completed! 1407 cells processed successfully, 7 errors
Significant cells (Holm-Bonferroni corrected, p<0.05): 92
Number of significant cells for condition #2: 92


,cell_ID,stat,p_value,error,p_value_holm_bonferroni,significant_holm,regular_rejected
0,11,0.365500,0.714738,None,1.000000,False,False
1,15,0.362962,0.716633,None,1.000000,False,False
2,16,-1.337628,0.181018,None,1.000000,False,False
3,17,1.101576,0.270646,None,1.000000,False,False
4,18,-1.187875,0.234883,None,1.000000,False,False
5,19,-1.604138,0.108684,None,1.000000,False,False
6,20,-0.827451,0.407981,None,1.000000,False,False
7,21,0.761458,0.446383,None,1.000000,False,False
8,23,4.254282,0.000021,None,0.027704,True,True
9,24,0.870600,0.383972,None,1.000000,False,False


In [32]:
len(cond_2_significant_cell_ids.intersection(significant_cell_ids))

29